import necessary libraries

In [ ]:
import pandas as pd
import numpy as np

Load raw dataset

In [ ]:
file = r"C:\Users\Lenovo\OneDrive\Documents\DataSprint\data\DataWave.csv"

df = pd.read_csv(file)


Inspect dataset


In [ ]:
df.isnull().sum()

df.info()
df.describe()
df.head()
df.tail()
df.columns

cleaning user_id

In [ ]:
df['user_id'].isnull().sum()
df = df.drop_duplicates(subset=['user_id'])

df= df.dropna(subset=['user_id'])


Convert all categorical text column to lower case and strip whitespaces. These are for the rows of each their respective columns. Ensure column is treated as text with the help of .astype(str).

In [ ]:
cat_cols = ['country', 'gender', 'subscription_type', 'churned']
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()

Cleaning gender values

In [ ]:
df['gender'] = df['gender'].replace({
    'm': 'male',
    'f': 'female',
    'femle' : 'female',
    'female ': 'female',
    'others': 'other',
    'other ': 'other',
    'prefer not to say': 'unknown',
    'nan': np.nan
})

Cleaning churned columns. Convert yes/no or text to 0/1 integer  [user left => 1, user stayed => 0]

In [ ]:
mapping = {
    'yes': 1, 'y': 1, '1': 1, 'true': 1,
    'no': 0, 'n': 0, '0': 0, 'false': 0
}

df['churned'] = (
    df['churned']
    .astype(str).str.strip().str.lower()
    .map(mapping)
)


# df['churned'].mean()   # Finding the mean of churned

Cleaning(Fix) Subscription Type

In [ ]:
df['subscription_type'] = df['subscription_type'].replace({
    'premuim': 'premium',
    'premum': 'premium',
    'free trial': 'free',
    'fam': 'family',
    'stud': 'student',
    'studnt': 'student'
})

Cleaning country

In [ ]:
df['country'] = df['country'].replace({
    'u.k.': 'United kingdom',
    'uk': 'United Kingdom',
    'U.K': ' United Kingdom',
    'united kingdom': 'United Kingdom',
    'United kingdom': 'United Kingdom',
    'ind': 'India',
    'india': 'India',
    'usa': 'United States',
    'us': 'United States',
    'nepal': 'Nepal',
    'nigeria': 'Nigeria',
    'ghana': 'Ghana',
    'kenya': 'Kenya',
    'brazil': 'Brazil',
    'south africa': 'South Africa'
})

Clean numeric columns. Convert invalid text like "ten" to NaN

In [ ]:
num_cols = ['age', 'avg_listening_hours_per_week', 'total_songs_played', 'skip_rate', 'satisfaction_score', 'monthly_fee']

for cols in num_cols:
    df[cols] = pd.to_numeric(df[cols], errors='coerce')   # convert invalid values to NaN)

CLEAN join_date. Convert to datetime.

In [ ]:
df['join_date'] = pd.to_datetime(df['join_date'], format='%m/%d/%Y', errors='coerce')
# df['join_date'].isnull().sum()

Checking Missing values

In [ ]:
# df.isnull().sum()

IMPUTE MISSING VALUES PROPERLY [Fill in the missing data in the correct or appropriate way.]
1. Gender (leave as NaN or fill with "unknown"). [fillna means to fill in missing values]

In [ ]:
df['gender'] = df['gender'].fillna('unknown')

2. When analyzing the skip rate, it's important to use the median rather than the mean. This is because the skip rate data is often skewed, meaning it can have outliers that disproportionately affect the average. The median provides a more accurate representation of the typical skip rate in such cases.

In [ ]:
df['skip_rate'] = df['skip_rate'].fillna(df["skip_rate"].median())

3. Fill missing monthly fee values using the median value specific to each subscription type not the overall median.

In [ ]:
df['monthly_fee'] = (
    df.groupby('subscription_type')['monthly_fee'].transform(lambda x: x.fillna(x.median()))
    )

4. Fill missing values for join_date

In [ ]:
df['join_date'] = df['join_date'].fillna(df['join_date'].mode()[0])

Remove or fix numeric outliers:
1. skip_rate must be between 0 and 1 

In [ ]:
df = df[(df['skip_rate'] >= 0) & (df['skip_rate'] <= 1)]

2. Age should be between 10 and 100

In [ ]:
df = df[(df['age'] >= 10) & (df['age'] <= 100)]

3. Max listening hours per week = 168 (24*7)

In [ ]:
df = df[df['avg_listening_hours_per_week'] <= 168]   

4. Satisfaction must be between 1 and 10

In [ ]:
df = df[(df['satisfaction_score'] >= 1) & (df['satisfaction_score'] <= 10)]

5. Songs played cannot be negative + cap unrealistic values

In [ ]:
df = df[df['total_songs_played'] >= 0]
df = df[df['total_songs_played'] < df['total_songs_played'].quantile(0.999)]

6.  Monthly fee cannot be negative or extremely high

In [ ]:
df = df[df['monthly_fee'] >= 0]
df = df[df['monthly_fee'] < df['monthly_fee'].quantile(0.999)]

Reset index after removing rows

In [ ]:
df = df.reset_index(drop=True)

FINAL MISSING VALUE CHECK

In [ ]:
print("Final missing values:")
print(df.isnull().sum())
print(df.describe())
print(df.info())

cleaned dataset

In [ ]:
cleaned_dataset = r"C:\Users\Lenovo\OneDrive\Documents\DataSprint\data\cleaned_DataWave.csv"
df.to_csv(cleaned_dataset, index=False)

df = pd.read_csv(cleaned_dataset)

UNIVARIATE ANALYSIS (Single Variable at a Time)
1. Weekly listening hours distribution [Summary statistics of average listening hours per week]

In [ ]:
df['avg_listening_hours_per_week'].describe() 

2. Satisfaction Score Description

In [ ]:
df['satisfaction_score'].describe()

3. Counting the number of each subscription type

In [ ]:
df['subscription_type'].value_counts()   

4. Counting the number of each country

In [ ]:
df['country'].value_counts()

5. Counting the number of each country

In [ ]:
df['gender'].value_counts()

Relationship insights [Analysis part] [BIVARIATE ANALYSIS (Two Variables Relationship)]
1. Does churn differ by subscription type?(Churn rate by subscription type)

In [ ]:
df.groupby('subscription_type')['churned'].mean().sort_values(ascending=False)

2. Do heavy listeners churn less? (Listening hours: churned vs active). Low hours = more churn, High hours = more engagement

In [ ]:
df.groupby('churned')['avg_listening_hours_per_week'].mean()

3. Does satisfaction predict churn?(Satisfaction score: churned vs active). If churned users have lower scores then obvious red flag for company. 


In [ ]:
df.groupby('churned')['satisfaction_score'].mean()

4. Which country or region has highest churn? (Top 10 countries with highest churn)

In [ ]:
df.groupby('country')['churned'].mean().sort_values(ascending=False).head(10)

5. Satisfaction by subscription type

In [ ]:
df.groupby('subscription_type')['satisfaction_score'].mean().sort_values()

6. Average listening hours by subscription type

In [ ]:
df.groupby('subscription_type')['avg_listening_hours_per_week'].mean().sort_values()

7. Churn rate by age group.
pd.cut takes a continuous variable (like age) and splits it into buckets (bins).
EG:-
age_group
(10, 20]     0.08
(20, 30]     0.12
(30, 40]     0.15
(40, 50]     0.09
(50, 60]     0.07
(60, 100]    0.20
Name: churned, dtype: float64


In [ ]:
df['age_group'] = pd.cut(df['age'], bins=[10,20,30,40,50,60,100])
df.groupby('age_group', observed=False)['churned'].mean()


8. Engagement vs churn (songs played)

In [ ]:
df.groupby('churned')['total_songs_played'].mean()

[MULTIVARIATE ANALYSIS (3 or More Variables Together)]

1. Correlation between numeric columns (to see drivers of churn). 

Correlation means How Two Things Move Together.
Imagine two things: X and Y (like age and monthly_charges).
Correlation tells you: When X goes up, does Y tend to go up as well? Or go down? Or not change in any clear way?

+1 → Perfect positive correlation, –1 → Perfect negative correlation, 0 → No clear linear relationship

For eg:-
It checks correlation between:-
Correlation between churned and age
Correlation between churned and monthly charges

In [ ]:
df.corr(numeric_only=True)['churned'].sort_values(ascending=False)

Visualization
1. Visualize churn rate by subscription type

In [ ]:
df.groupby('subscription_type')['churned'].mean().plot(kind='bar')

2. Visualize distribution of satisfaction score

In [ ]:
df['satisfaction_score'].plot(kind='hist', bins=20)

3. Boxplot: Listening hours by churn

In [ ]:
df.boxplot(column='avg_listening_hours_per_week', by='churned')

4. Heatmap for numeric correlation

In [ ]:
import seaborn as sns
sns.heatmap(df.corr(numeric_only=True), annot=True)

5. Country-wise churn bar plot (Top 10)

In [ ]:
df.groupby('country')['churned'].mean().sort_values().tail(10).plot(kind='barh')

6. Scatter: Satisfaction vs Listening Hours

In [ ]:
df.plot(kind='scatter', x='satisfaction_score', y='avg_listening_hours_per_week')

7. Create churn summary table

In [ ]:
df.groupby('churned').mean(numeric_only=True)

Build a Churn Prediction Model (Machine Learning)
MODEL BUILDING
1. Check class balance (churned vs not churned)

In [ ]:
df['churned'].value_counts(normalize=True)

2. Select the useful columns for modeling

In [ ]:
features = [
    'age',
    'gender',
    'subscription_type',
    'avg_listening_hours_per_week',
    'total_songs_played',
    'skip_rate',
    'satisfaction_score',
    'monthly_fee',
    'country'
]

X = df[features]
y = df['churned']


3. Convert all categorical columns into numbers (One-Hot Encoding)

In [ ]:
X = pd.get_dummies(X, drop_first=True)

4. Split data into Train and Test sets

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

5. Scale numeric features

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


BASELINE MODELS
6. Train Logistic Regression (Baseline model)

In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=3000, solver='lbfgs')
log_reg.fit(X_train_scaled, y_train)


MODEL EVALUATION
1. Evaluate Logistic Regression

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred_lr = log_reg.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))


TREE-BASED MODELS (Better Accuracy)
2. Train Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

3. Evaluate Random Forest

In [ ]:
y_pred_rf = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


FEATURE IMPORTANCE
4. Find which features influence churn the most

In [ ]:
importances = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

importances.head(15)


MODEL COMPARISON
5. Compare model performances

In [ ]:
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

6. Add Confusion Matrix for Both Models

In [ ]:
from sklearn.metrics import confusion_matrix

print("LR Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

print("RF Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))


7. Add ROC Curve & AUC (VERY IMPORTANT for churn)

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

# Probabilities
lr_probs = log_reg.predict_proba(X_test_scaled)[:,1]
rf_probs = rf.predict_proba(X_test)[:,1]

print("LR AUC:", roc_auc_score(y_test, lr_probs))
print("RF AUC:", roc_auc_score(y_test, rf_probs))


8. Plot ROC Curves

In [ ]:
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_probs)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_probs)

plt.plot(fpr_lr, tpr_lr, label="Logistic Regression")
plt.plot(fpr_rf, tpr_rf, label="Random Forest")
plt.plot([0,1], [0,1], 'k--')
plt.legend()
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.show()

Handle Class Imbalance Properly

In [ ]:
df['churned'].value_counts(normalize=True)

Train a Tuned Random Forest (Hyperparameter Search)

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    scoring='f1',
    cv=3,
    n_jobs=-1
)

grid.fit(X_train, y_train)

best_rf = grid.best_estimator_
best_rf


Evaluate the Tuned Model

In [ ]:
y_pred_best = best_rf.predict(X_test)

print("Tuned RF Accuracy:", accuracy_score(y_test, y_pred_best))
print(classification_report(y_test, y_pred_best))


INSIGHTS & BUSINESS ANALYSIS
1. Create Final Churn Insights Summary (Top Drivers)

In [ ]:
importances = pd.Series(
    best_rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

importances.head(10)

2. Convert Feature Importance into Human-Readable Insights

In [ ]:
important_features = importances.head(10)
important_features

3. Build a Churn Probability Column for Each User

In [ ]:
df['churn_probability'] = best_rf.predict_proba(X)[:,1]
df[['user_id', 'churn_probability']].head()

4. Label High-Risk Users (e.g., >0.6 probability)

In [ ]:
df['risk_label'] = df['churn_probability'].apply(
    lambda x: 'high_risk' if x > 0.6 else 'low_risk'
)
df['risk_label'].value_counts()

5. Create a Segment Table (Subscription × Country × Risk)

In [ ]:
segment_table = df.groupby(['subscription_type', 'country', 'risk_label']).size()
segment_table

6. Create an Actionable “Why they churn” Table

In [ ]:
df.groupby('churned').mean(numeric_only=True)

7. Build a Retention Strategy Table (Code Version)

In [ ]:
retention_factors = importances.head(10)
retention_factors

DEPLOYMENT PREPARATION 
1. Save the Final Model for Deployment

In [ ]:
import joblib

joblib.dump(best_rf, "final_churn_model.pkl")
joblib.dump(scaler, "scaler.pkl")


2. Save Enhanced Dataset (with churn probability + risk label)

In [ ]:
df.to_csv("DataWave_Final_With_Risk.csv", index=False)

3. Build a Simple Prediction Function

In [ ]:
def predict_churn(new_data):
    new_data = pd.get_dummies(new_data, drop_first=True)
    new_data = new_data.reindex(columns=X.columns, fill_value=0)
    new_data_scaled = scaler.transform(new_data)
    return best_rf.predict_proba(new_data_scaled)[:,1]


Create proper, polished visualizations
1. Churn rate by subscription type(Bar Chart)

In [ ]:
plt.style.use('seaborn-v0_8')

In [ ]:
sub_churn = df.groupby('subscription_type')['churned'].mean().sort_values()

plt.figure(figsize=(8,5))
sns.barplot(x=sub_churn.index, y=sub_churn.values)
plt.title("Churn Rate by Subscription Type")
plt.ylabel("Churn Rate")
plt.xlabel("Subscription Type")
plt.show()


2. Satisfaction Score Distribution (Histogram)

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['satisfaction_score'], bins=20, kde=True)
plt.title("Distribution of Satisfaction Score")
plt.xlabel("Satisfaction Score")
plt.ylabel("Count")
plt.show()

3. Listening Hours: Churned vs Active (Boxplot)

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='churned', y='avg_listening_hours_per_week')
plt.title("Listening Hours: Churned vs Active Users")
plt.xlabel("Churned (0 = stayed, 1 = left)")
plt.ylabel("Listening Hours per Week")
plt.show()

4. Top 10 Countries by Churn (Horizontal Bar Chart)

In [ ]:
country_churn = df.groupby('country')['churned'].mean().sort_values().tail(10)

plt.figure(figsize=(8,6))
sns.barplot(y=country_churn.index, x=country_churn.values)
plt.title("Top 10 Countries by Churn Rate")
plt.xlabel("Churn Rate")
plt.ylabel("Country")
plt.show()


5. Correlation Heatmap

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

6. Scatter Plot: Satisfaction vs Listening Hours

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(
    data=df,
    x='satisfaction_score',
    y='avg_listening_hours_per_week',
    hue='churned'
)
plt.title("Satisfaction vs Listening Hours")
plt.show()


7. Boxplot of Satisfaction by Subscription Type

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='subscription_type', y='satisfaction_score')
plt.title("Satisfaction Score by Subscription Type")
plt.show()

8. Save cleaned dataset again

In [ ]:
df.to_csv("cleaned_with_visuals_ready.csv", index=False)

Add ROC Curve Preparation

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

lr_probs = log_reg.predict_proba(X_test_scaled)[:,1]
rf_probs = rf.predict_proba(X_test)[:,1]

fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_probs)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_probs)


Add ROC Curve Plot

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(fpr_lr, tpr_lr, label="Logistic Regression")
plt.plot(fpr_rf, tpr_rf, label="Random Forest")
plt.plot([0,1],[0,1],'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

Evaluate Models Properly (Final Evaluation Block)

In [ ]:
# Final Evaluation Summary

from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(name, y_true, y_pred, y_prob):
    print(f"\n===== {name} =====")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))
    print("F1 Score:", f1_score(y_true, y_pred))
    print("AUC:", roc_auc_score(y_true, y_prob))
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))


# Logistic Regression Evaluation
evaluate_model("Logistic Regression",
               y_test, 
               y_pred_lr, 
               lr_probs)

# Random Forest Evaluation
evaluate_model("Random Forest",
               y_test, 
               y_pred_rf, 
               rf_probs)

# Tuned Random Forest Evaluation
best_rf_probs = best_rf.predict_proba(X_test)[:,1]
best_rf_pred = best_rf.predict(X_test)

evaluate_model("Tuned Random Forest",
               y_test,
               best_rf_pred,
               best_rf_probs)


Save Final Visualizations as PNG Files for 6 key charts.

In [ ]:
import os

# Create folder for saving visuals
os.makedirs("final_visuals", exist_ok=True)

# 1. Churn Rate by Subscription Type
plt.figure(figsize=(8,5))
sns.barplot(x=sub_churn.index, y=sub_churn.values)
plt.title("Churn Rate by Subscription Type")
plt.savefig("final_visuals/churn_by_subscription.png", dpi=300)
plt.close()


# 2. Satisfaction Score Distribution
plt.figure(figsize=(8,5))
sns.histplot(df['satisfaction_score'], bins=20, kde=True)
plt.title("Distribution of Satisfaction Score")
plt.xlabel("Satisfaction Score")
plt.ylabel("Count")
plt.savefig("final_visuals/satisfaction_distribution.png", dpi=300)
plt.close()

# 3. Listening Hours: Churned vs Active (Boxplot)
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='churned', y='avg_listening_hours_per_week')
plt.title("Listening Hours: Churned vs Active Users")
plt.xlabel("Churned (0 = stayed, 1 = left)")
plt.ylabel("Listening Hours per Week")
plt.savefig("final_visuals/listening_hours_boxplot.png", dpi=300)
plt.close()

# 4. Top 10 Countries by Churn Rate
country_churn = df.groupby('country')['churned'].mean().sort_values().tail(10)
plt.figure(figsize=(8,6))
sns.barplot(y=country_churn.index, x=country_churn.values)
plt.title("Top 10 Countries by Churn Rate")
plt.xlabel("Churn Rate")
plt.ylabel("Country")
plt.savefig("final_visuals/top10_country_churn.png", dpi=300)
plt.close()

# 5. Correlation Heatmap
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.savefig("final_visuals/correlation_heatmap.png", dpi=300)
plt.close()

# 6. ROC Curve
from sklearn.metrics import roc_curve

lr_probs = log_reg.predict_proba(X_test_scaled)[:,1]
rf_probs = rf.predict_proba(X_test)[:,1]

fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_probs)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_probs)

plt.figure(figsize=(8,5))
plt.plot(fpr_lr, tpr_lr, label="Logistic Regression")
plt.plot(fpr_rf, tpr_rf, label="Random Forest")
plt.plot([0,1],[0,1],'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.savefig("final_visuals/roc_curve.png", dpi=300)
plt.close()


Create a Short Final Summary Table

In [ ]:
summary_table = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1", "AUC"],
    "Logistic Regression": [
        accuracy_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_lr),
        roc_auc_score(y_test, lr_probs)
    ],
    "Random Forest": [
        accuracy_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_rf),
        roc_auc_score(y_test, rf_probs)
    ],
    "Tuned Random Forest": [
        accuracy_score(y_test, best_rf_pred),
        precision_score(y_test, best_rf_pred),
        recall_score(y_test, best_rf_pred),
        f1_score(y_test, best_rf_pred),
        roc_auc_score(y_test, best_rf_probs)
    ]
})

summary_table
